# XGBoost

**Gradient boosting** builds a model from many small trees, each one fixing the mistakes of the trees before it. **XGBoost** is a fast, well-tuned library for it. The lesson page shows a simplified version; this notebook uses the real library.

This notebook goes with the **XGBoost** lesson on [ML Visual](https://mohammad-qt.github.io/ML_DL_Vis/). Run each cell in order with **Shift + Enter**. Nothing to install: Colab already has every library used here.

## 1. Load the data

Each row is a tumour sample from a breast biopsy. The columns are eight measurements of the cell nuclei (size, texture, shape). The goal: predict whether the sample is **malignant** (`diagnosis = 1`) or **benign** (`0`).

The data is the Breast Cancer Wisconsin (Diagnostic) data set from the UCI repository (CC BY 4.0), reduced to eight columns. This is a teaching example, not medical advice.

In [ ]:
import numpy as np
import pandas as pd

# The data lives in this project's GitHub repository, so there is nothing to upload.
DATA_URL = "https://raw.githubusercontent.com/Mohammad-QT/ML_DL_Vis/main/datasets/breast_cancer.csv"
dataset = pd.read_csv(DATA_URL)
X = dataset.drop(columns="diagnosis")
y = dataset["diagnosis"]
print(dataset.shape)
dataset.head()

## 2. Split the data

We need three parts: training data, a **validation** set that decides when to stop, and a final test set.

In [ ]:
from sklearn.model_selection import train_test_split

X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=0, stratify=y_rest)
print(len(X_train), len(X_val), len(X_test))

## 3. Train with early stopping

- `n_estimators` is the maximum number of rounds (trees).
- `learning_rate` is how big a step each tree takes.
- `max_depth` is how deep each tree may grow.
- `early_stopping_rounds=10` stops when the validation loss has not improved for 10 rounds.

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300, learning_rate=0.1, max_depth=3,
    eval_metric="logloss", early_stopping_rounds=10, random_state=0,
)
model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)
print("best round:", model.best_iteration)
print(f"test accuracy: {model.score(X_test, y_test):.3f}")

## 4. Plot the curves

The training loss keeps falling. The validation loss stops improving. That is the picture behind early stopping.

In [ ]:
import matplotlib.pyplot as plt

hist = model.evals_result()
plt.plot(hist["validation_0"]["logloss"], label="train")
plt.plot(hist["validation_1"]["logloss"], label="validation")
plt.axvline(model.best_iteration, color="grey", linestyle="--", label="best round")
plt.xlabel("round")
plt.ylabel("log loss (lower is better)")
plt.legend()
plt.show()

## 5. Which measurements matter?

In [ ]:
pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

## 6. Tune it with cross-validation

Combine this lesson with the last two: a small grid search over the learning rate and the depth.

In [ ]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    XGBClassifier(n_estimators=100, random_state=0),
    {"learning_rate": [0.05, 0.1, 0.3], "max_depth": [2, 3, 5]},
    cv=5,
)
search.fit(X_rest, y_rest)
print(search.best_params_, round(search.best_score_, 3))

## 7. Try it yourself

- Set `learning_rate=1.0`. Does the validation loss get worse sooner?
- Set `max_depth=8`. What happens to the gap between train and validation?